# FHRSS+FCPE: Memorie Infinita + Qwen2.5-72B

**Scriere de Romane cu Context Infinit**

- **Model:** Qwen2.5-72B-Instruct (Apache 2.0 - 100% FREE)
- **VRAM:** ~45GB (ai 80GB - perfect!)
- **Limba:** Romana
- **Memorie:** FHRSS+FCPE - 2M+ tokens, stocare in Google Drive

**Capabilitati:**
- Scrie romane complete cu consistenta perfecta
- Memoreaza personaje, locatii, evenimente
- 100% recovery la 40% data loss
- Fara API key, totul ruleaza local

---
*Patent: EP25216372.0 (FHRSS - OmniVault)*  
*Author: Vasile Lucian Borbeleac*

## 1. Setup: Mount Google Drive & Install Dependencies

In [ ]:
# Mount Google Drive pentru stocare persistenta
from google.colab import drive
drive.mount('/content/drive')

# Creeaza directorul pentru FHRSS storage
import os
STORAGE_PATH = '/content/drive/MyDrive/FHRSS_FCPE_Colab_OpenSource'
os.makedirs(STORAGE_PATH, exist_ok=True)
print(f"Storage path: {STORAGE_PATH}")

In [ ]:
# Instalare dependente
!pip install -q sentence-transformers numpy transformers accelerate bitsandbytes
print("Dependencies installed")

## 2. FHRSS+FCPE Core Implementation

In [ ]:
import numpy as np
import hashlib
import pickle
import time
import json
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass, field
from functools import reduce
from operator import xor

@dataclass
class FCPEConfig:
    dim: int = 384
    num_layers: int = 5
    lambda_s: float = 0.5
    phi: float = 1.618033988749895
    compression_method: str = "weighted_attention"
    use_whitening: bool = True
    use_content_seed: bool = True
    jitter_scale: float = 0.05

@dataclass
class FHRSSConfig:
    subcube_size: int = 8
    profile: str = "FULL"
    use_checksums: bool = True

@dataclass
class UnifiedConfig:
    fcpe: FCPEConfig = field(default_factory=FCPEConfig)
    fhrss: FHRSSConfig = field(default_factory=FHRSSConfig)
    storage_path: str = "./fhrss_storage"
    auto_persist: bool = True

print("Configuration classes defined")

In [ ]:
class FCPEEncoder:
    """Fractal-Chaotic Persistent Encoding"""

    def __init__(self, config: FCPEConfig):
        self.config = config
        self.dim = config.dim
        self.num_layers = config.num_layers
        self.lambda_s = config.lambda_s
        self.phi = config.phi
        self.transforms = self._generate_transforms()
        self.permutations = self._generate_permutations()

    def _generate_transforms(self):
        transforms = []
        for i in range(self.num_layers):
            seed = int((i + 1) * self.phi * 1000000) % (2**31)
            np.random.seed(seed)
            W = np.random.randn(self.dim, self.dim)
            U, _, Vt = np.linalg.svd(W)
            transforms.append((U @ Vt).astype(np.float32))
        return transforms

    def _generate_permutations(self):
        permutations = []
        for i in range(self.num_layers):
            seed = int((i + 1) * self.phi * 2000000) % (2**31)
            np.random.seed(seed)
            perm = np.random.permutation(self.dim)
            permutations.append(perm)
        return permutations

    def _content_hash(self, seq):
        sig = np.concatenate([
            seq.mean(axis=0)[:16],
            seq.std(axis=0)[:16],
            seq[0][:16] if len(seq) > 0 else np.zeros(16),
            seq[-1][:16] if len(seq) > 0 else np.zeros(16),
        ])
        return int(hashlib.md5(sig.tobytes()).hexdigest(), 16) % (2**31)

    def encode(self, embeddings):
        if embeddings.ndim == 1:
            embeddings = embeddings.reshape(1, -1)
        if embeddings.ndim == 2:
            return self._encode_sequence(embeddings)
        elif embeddings.ndim == 3:
            return np.stack([self._encode_sequence(seq) for seq in embeddings])

    def _encode_sequence(self, seq):
        if self.config.use_whitening:
            mean = seq.mean(axis=0)
            std = seq.std(axis=0)
            std = np.where(std < 1e-5, 1.0, std)
            seq = (seq - mean) / std

        if self.config.compression_method == "weighted_attention":
            norms = np.linalg.norm(seq, axis=1)
            mean_vec = seq.mean(axis=0)
            deviations = np.linalg.norm(seq - mean_vec, axis=1)
            scores = norms * (1 + deviations)
            scores = scores - scores.max()
            weights = np.exp(scores)
            weights = weights / (weights.sum() + 1e-8)
            x = (weights[:, None] * seq).sum(axis=0)
        else:
            x = seq.mean(axis=0)

        if len(x) != self.dim:
            np.random.seed(42)
            proj = np.random.randn(len(x), self.dim) / np.sqrt(len(x))
            x = x @ proj

        if self.config.use_content_seed:
            content_hash = self._content_hash(seq)
            rng = np.random.default_rng(content_hash)
            jitter = rng.standard_normal(self.dim) * self.config.jitter_scale
            x = x + jitter

        for i in range(self.num_layers):
            h = x @ self.transforms[i]
            h = h[self.permutations[i]]
            x = self.lambda_s * x + (1 - self.lambda_s) * h

        x = x / (np.linalg.norm(x) + 1e-8)
        return x.astype(np.float32)

print("FCPE Encoder defined")

In [ ]:
class FHRSSEncoder:
    """Fractal-Holographic Redundant Storage System"""

    PROFILES = {
        "MINIMAL": ["X", "Y", "Z"],
        "FULL": ["X", "Y", "Z", "DXYp", "DXYn", "DXZp", "DXZn", "DYZp", "DYZn"]
    }
    RECOVERY_PRIORITY = ["X", "Y", "Z", "DXYp", "DXZp", "DYZp", "DXYn", "DXZn", "DYZn"]

    def __init__(self, config: FHRSSConfig):
        self.config = config
        self.m = config.subcube_size
        self.families = self.PROFILES[config.profile]
        self.num_families = len(self.families)
        self._line_cache = {}
        for family in self.RECOVERY_PRIORITY:
            self._line_cache[family] = self._compute_line_indices(family)

    def _compute_line_indices(self, family):
        m = self.m
        lines = []
        if family == "X":
            for y in range(m):
                for z in range(m):
                    lines.append([(x, y, z) for x in range(m)])
        elif family == "Y":
            for x in range(m):
                for z in range(m):
                    lines.append([(x, y, z) for y in range(m)])
        elif family == "Z":
            for x in range(m):
                for y in range(m):
                    lines.append([(x, y, z) for z in range(m)])
        elif family.startswith("D"):
            for i in range(m):
                for k in range(m):
                    if family == "DXYp":
                        lines.append([(j, (j + k) % m, i) for j in range(m)])
                    elif family == "DXYn":
                        lines.append([(j, (k - j) % m, i) for j in range(m)])
                    elif family == "DXZp":
                        lines.append([(j, i, (j + k) % m) for j in range(m)])
                    elif family == "DXZn":
                        lines.append([(j, i, (k - j) % m) for j in range(m)])
                    elif family == "DYZp":
                        lines.append([(i, j, (j + k) % m) for j in range(m)])
                    elif family == "DYZn":
                        lines.append([(i, j, (k - j) % m) for j in range(m)])
        return lines

    def encode(self, data: bytes):
        m = self.m
        subcube_bytes = m ** 3
        num_subcubes = (len(data) + subcube_bytes - 1) // subcube_bytes
        padded = data + b'\x00' * (num_subcubes * subcube_bytes - len(data))

        encoded_subcubes = []
        for sc_id in range(num_subcubes):
            start = sc_id * subcube_bytes
            chunk = padded[start:start + subcube_bytes]
            cube = np.frombuffer(chunk, dtype=np.uint8).reshape(m, m, m).copy()
            checksum = hashlib.sha256(chunk).hexdigest() if self.config.use_checksums else None
            parity = {}
            for family in self.families:
                parity[family] = self._compute_family_parity(cube, family)
            encoded_subcubes.append({
                'data': cube.tobytes(),
                'parity': parity,
                'checksum': checksum,
                'subcube_id': sc_id
            })

        return {'subcubes': encoded_subcubes, 'original_length': len(data)}

    def _compute_family_parity(self, cube, family):
        lines = self._line_cache[family]
        return [reduce(xor, [int(cube[x, y, z]) for x, y, z in line], 0) for line in lines]

    def decode(self, encoded, loss_masks=None):
        m = self.m
        recovered_data = []
        for idx, sc in enumerate(encoded['subcubes']):
            cube = np.frombuffer(sc['data'], dtype=np.uint8).reshape(m, m, m).copy()
            if loss_masks and idx < len(loss_masks):
                cube = self._recover_subcube(cube, sc['parity'], loss_masks[idx])
            recovered_data.append(cube.tobytes())
        return b''.join(recovered_data)[:encoded['original_length']]

    def _recover_subcube(self, data, parity, loss_mask):
        m = self.m
        data = data.copy()
        data[loss_mask] = 0
        recovered_mask = ~loss_mask
        for _ in range(self.num_families * 2):
            recovered_this_pass = 0
            for family in self.RECOVERY_PRIORITY:
                if family not in parity:
                    continue
                for line_idx, line_indices in enumerate(self._line_cache[family]):
                    missing = [(x, y, z) for x, y, z in line_indices if not recovered_mask[x, y, z]]
                    if len(missing) == 1:
                        x, y, z = missing[0]
                        present = [data[px, py, pz] for px, py, pz in line_indices if recovered_mask[px, py, pz]]
                        data[x, y, z] = parity[family][line_idx] ^ reduce(xor, present, 0)
                        recovered_mask[x, y, z] = True
                        recovered_this_pass += 1
            if recovered_this_pass == 0:
                break
        return data

print("FHRSS Encoder defined")

In [ ]:
@dataclass
class EncodedContext:
    context_id: int
    fcpe_vector: np.ndarray
    fhrss_encoded: Dict[str, Any]
    original_hash: str
    metadata: Dict[str, Any]
    timestamp: float
    access_count: int = 0


class UnifiedFHRSS_FCPE:
    """Unified FHRSS + FCPE System"""

    def __init__(self, config: UnifiedConfig = None):
        self.config = config or UnifiedConfig()
        self.fcpe = FCPEEncoder(self.config.fcpe)
        self.fhrss = FHRSSEncoder(self.config.fhrss)
        self.storage_path = Path(self.config.storage_path)
        self.storage_path.mkdir(parents=True, exist_ok=True)
        self.contexts: Dict[int, EncodedContext] = {}
        self.next_id = 0
        self._load_from_disk()
        print(f"UnifiedFHRSS_FCPE initialized: {len(self.contexts)} contexts loaded")

    def encode_context(self, embeddings, metadata=None, store_original=True):
        if embeddings.ndim == 1:
            embeddings = embeddings.reshape(1, -1)
        if store_original and embeddings.shape[0] <= 3:
            fcpe_vector = embeddings.mean(axis=0)
            fcpe_vector = fcpe_vector / (np.linalg.norm(fcpe_vector) + 1e-8)
        else:
            fcpe_vector = self.fcpe.encode(embeddings)
        fcpe_bytes = fcpe_vector.astype(np.float32).tobytes()
        fhrss_encoded = self.fhrss.encode(fcpe_bytes)
        original_hash = hashlib.sha256(fcpe_bytes).hexdigest()
        ctx_id = self.next_id
        self.next_id += 1
        context = EncodedContext(
            context_id=ctx_id,
            fcpe_vector=fcpe_vector,
            fhrss_encoded=fhrss_encoded,
            original_hash=original_hash,
            metadata=metadata or {},
            timestamp=time.time()
        )
        self.contexts[ctx_id] = context
        if self.config.auto_persist:
            self._persist_context(context)
        return ctx_id

    def retrieve_similar(self, query, top_k=5):
        if not self.contexts:
            return []
        query = query / (np.linalg.norm(query) + 1e-8)
        similarities = []
        for ctx_id, ctx in self.contexts.items():
            vec = ctx.fcpe_vector
            sim = np.dot(query, vec / (np.linalg.norm(vec) + 1e-8))
            similarities.append((ctx_id, float(sim)))
        similarities.sort(key=lambda x: x[1], reverse=True)
        return [{'ctx_id': cid, 'similarity': sim, 'metadata': self.contexts[cid].metadata}
                for cid, sim in similarities[:top_k]]

    def _persist_context(self, context):
        path = self.storage_path / f"ctx_{context.context_id}.pkl"
        data = {
            'context_id': context.context_id,
            'fcpe_vector': context.fcpe_vector.tobytes(),
            'fhrss_encoded': context.fhrss_encoded,
            'original_hash': context.original_hash,
            'metadata': context.metadata,
            'timestamp': context.timestamp
        }
        with open(path, 'wb') as f:
            pickle.dump(data, f)

    def _load_from_disk(self):
        for path in self.storage_path.glob("ctx_*.pkl"):
            try:
                with open(path, 'rb') as f:
                    data = pickle.load(f)
                fcpe_vector = np.frombuffer(data['fcpe_vector'], dtype=np.float32)
                context = EncodedContext(
                    context_id=data['context_id'],
                    fcpe_vector=fcpe_vector,
                    fhrss_encoded=data['fhrss_encoded'],
                    original_hash=data['original_hash'],
                    metadata=data.get('metadata', {}),
                    timestamp=data['timestamp']
                )
                self.contexts[context.context_id] = context
                self.next_id = max(self.next_id, context.context_id + 1)
            except Exception as e:
                print(f"Warning: Failed to load {path}: {e}")

    def get_stats(self):
        return {'num_contexts': len(self.contexts), 'storage_path': str(self.storage_path)}

print("Unified FHRSS+FCPE System defined")

## 3. Initialize System with Google Drive Storage

In [ ]:
# Initializare sistem cu stocare in Google Drive
config = UnifiedConfig(
    fcpe=FCPEConfig(dim=384),
    fhrss=FHRSSConfig(profile="FULL"),
    storage_path=STORAGE_PATH,
    auto_persist=True
)

memory = UnifiedFHRSS_FCPE(config)
print(f"\nMemory Stats: {memory.get_stats()}")

In [ ]:
# Incarca modelul de embeddings
from sentence_transformers import SentenceTransformer

print("Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Embedding model loaded (dim={embedder.get_sentence_embedding_dimension()})")

## 4. Load Qwen2.5-32B (Optimal for 80GB VRAM)

**Model:** Qwen/Qwen2.5-32B-Instruct  
**License:** Apache 2.0 (100% FREE)  
**VRAM:** ~64GB (fits in 80GB)  
**Speed:** Fast  
**Quality:** Excellent for Romanian and creative writing

In [ ]:
# Qwen2.5-32B - Optimal pentru 80GB VRAM (Apache 2.0)
MODEL_ID = "Qwen/Qwen2.5-32B-Instruct"

print(f"Loading: {MODEL_ID}")
print("License: Apache 2.0 - 100% FREE")
print("VRAM: ~64GB (fits in 80GB)")

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Model - load in bfloat16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

print("Qwen2.5-32B loaded!")

In [ ]:
def get_system_info():
    """Returneaza info real despre sistem."""
    import torch
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
    
    return f"""SISTEM REAL:
- GPU: {gpu_name}
- VRAM Total: {vram:.1f} GB
- Model LLM: {MODEL_ID}
- Embedding: all-MiniLM-L6-v2 (384 dim)
- Memorie: FHRSS+FCPE (2M+ tokens, 100% recovery la 40% loss)
- Storage: {STORAGE_PATH}
- Contexte salvate: {len(memory.contexts)}"""

def save_to_memory(content, memory_type="note"):
    """Salveaza ceva important in memorie."""
    emb = embedder.encode(content, convert_to_numpy=True)
    ctx_id = memory.encode_context(
        emb.reshape(1, -1),
        metadata={'type': memory_type, 'content': content[:500]}
    )
    return ctx_id

# Incarca info sistem in memorie la start
system_info = get_system_info()
save_to_memory(system_info, memory_type="system_info")
print("System info salvat in memorie:")
print(system_info)


def generate_response(prompt, max_new_tokens=1024):
    """Genereaza raspuns cu Qwen2.5-32B in romana."""
    
    # Adauga system info la context
    sys_info = get_system_info()
    
    messages = [
        {"role": "system", "content": f"""Esti un asistent creativ de scriere. Raspunzi INTOTDEAUNA in limba romana.

{sys_info}

IMPORTANT: Daca utilizatorul iti spune ceva important despre el (nume, profesie, preferinte, proiecte), 
raspunde cu [MEMOREZ: informatia] la sfarsitul mesajului pentru a salva in memorie.
Exemple: [MEMOREZ: Utilizatorul se numeste Ion si e programator]"""},
        {"role": "user", "content": prompt}
    ]
    
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()


def chat_with_infinite_context(user_message, top_k=5):
    """Chat cu Qwen2.5-32B + FHRSS+FCPE pentru context infinit."""
    
    # 1. Embed user message
    user_embedding = embedder.encode(user_message, convert_to_numpy=True)
    
    # 2. Store user message
    memory.encode_context(
        user_embedding.reshape(1, -1),
        metadata={'role': 'user', 'content': user_message[:500]}
    )
    
    # 3. Retrieve relevant context
    relevant = memory.retrieve_similar(user_embedding, top_k=top_k)
    
    # 4. Build context
    context_parts = []
    for r in relevant:
        if r['similarity'] > 0.3:
            meta = r['metadata']
            ctx_type = meta.get('type', meta.get('role', 'unknown'))
            content = meta.get('content', '')
            context_parts.append(f"[{ctx_type}]: {content}")
    
    # 5. Build full prompt
    if context_parts:
        context_str = "\n".join(context_parts[-5:])
        full_prompt = f"""MEMORIE RELEVANTA:
{context_str}

MESAJ CURENT: {user_message}"""
    else:
        full_prompt = user_message
    
    # 6. Generate response
    response_text = generate_response(full_prompt)
    
    # 7. Extrage si salveaza memorii noi
    if "[MEMOREZ:" in response_text:
        import re
        memories = re.findall(r'\[MEMOREZ:\s*([^\]]+)\]', response_text)
        for mem in memories:
            save_to_memory(mem.strip(), memory_type="learned")
            print(f"   [Salvat in memorie: {mem.strip()[:50]}...]")
        # Sterge tagurile din raspuns
        response_text = re.sub(r'\[MEMOREZ:[^\]]+\]', '', response_text).strip()
    
    # 8. Store response in memory
    response_embedding = embedder.encode(response_text, convert_to_numpy=True)
    memory.encode_context(
        response_embedding.reshape(1, -1),
        metadata={'role': 'assistant', 'content': response_text[:500]}
    )
    
    return response_text

print("\nChat ready - AI poate vedea sistemul si salva memorii!")

## 5. Incarca Documente in Memorie

Poti incarca fisiere din:
- **Upload direct** in Colab
- **Google Drive** (deja montat)
- **URL** (descarca automat)

Formate suportate: **TXT, PDF, DOCX, MD, JSON, CSV, PY (Python)**

In [ ]:
# Instalare dependente pentru citire documente
!pip install -q PyPDF2 python-docx requests

import re
from google.colab import files as colab_files

def read_txt(filepath):
    """Citeste fisier TXT/MD/PY."""
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()

def read_pdf(filepath):
    """Citeste fisier PDF."""
    from PyPDF2 import PdfReader
    reader = PdfReader(filepath)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

def read_docx(filepath):
    """Citeste fisier DOCX."""
    from docx import Document
    doc = Document(filepath)
    return "\n".join([p.text for p in doc.paragraphs])

def read_json(filepath):
    """Citeste fisier JSON."""
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return json.dumps(data, indent=2, ensure_ascii=False)

def read_csv(filepath):
    """Citeste fisier CSV."""
    import csv
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        reader = csv.reader(f)
        rows = list(reader)
    return "\n".join([", ".join(row) for row in rows])

def read_document(filepath):
    """Citeste orice tip de document suportat."""
    filepath = str(filepath)
    ext = filepath.lower().split('.')[-1]
    
    readers = {
        'txt': read_txt, 'md': read_txt, 'text': read_txt,
        'py': read_txt, 'python': read_txt,  # Python files
        'js': read_txt, 'ts': read_txt,      # JavaScript/TypeScript
        'java': read_txt, 'cpp': read_txt, 'c': read_txt, 'h': read_txt,  # C/C++/Java
        'rs': read_txt, 'go': read_txt,      # Rust/Go
        'html': read_txt, 'css': read_txt, 'xml': read_txt,  # Web
        'yaml': read_txt, 'yml': read_txt, 'toml': read_txt,  # Config
        'sh': read_txt, 'bash': read_txt,    # Shell
        'sql': read_txt,                      # SQL
        'pdf': read_pdf,
        'docx': read_docx, 'doc': read_docx,
        'json': read_json,
        'csv': read_csv
    }
    
    if ext in readers:
        return readers[ext](filepath)
    else:
        # Incearca ca text
        return read_txt(filepath)

def chunk_text(text, chunk_size=500, overlap=50):
    """Imparte textul in bucati pentru embedding."""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
    return chunks

def load_document_to_memory(filepath, doc_name=None):
    """Incarca un document in memoria FHRSS+FCPE."""
    filepath = str(filepath)
    doc_name = doc_name or Path(filepath).name
    
    print(f"Citesc: {filepath}")
    text = read_document(filepath)
    print(f"   Lungime: {len(text)} caractere")
    
    # Chunk text
    chunks = chunk_text(text, chunk_size=300, overlap=30)
    print(f"   Chunks: {len(chunks)}")
    
    # Embed si stocheaza fiecare chunk
    for i, chunk in enumerate(chunks):
        emb = embedder.encode(chunk, convert_to_numpy=True)
        memory.encode_context(
            emb.reshape(1, -1),
            metadata={
                'type': 'document',
                'source': doc_name,
                'chunk_id': i,
                'content': chunk[:500]
            }
        )
    
    print(f"   Incarcat in memorie: {len(chunks)} chunks din '{doc_name}'")
    return len(chunks)

def upload_and_load():
    """Upload fisier din computer si incarca in memorie."""
    print("Selecteaza fisierul de uploadat...")
    uploaded = colab_files.upload()
    
    for filename in uploaded.keys():
        load_document_to_memory(filename, filename)
    
    print(f"\nTotal contexte in memorie: {memory.get_stats()['num_contexts']}")

def load_from_drive(path):
    """Incarca fisier din Google Drive."""
    full_path = f"/content/drive/MyDrive/{path}" if not path.startswith("/") else path
    return load_document_to_memory(full_path)

def load_from_url(url, doc_name=None):
    """Descarca si incarca fisier de la URL."""
    import requests
    doc_name = doc_name or url.split("/")[-1]
    
    print(f"Descarc: {url}")
    response = requests.get(url)
    
    temp_path = f"/tmp/{doc_name}"
    with open(temp_path, 'wb') as f:
        f.write(response.content)
    
    return load_document_to_memory(temp_path, doc_name)

print("Functii de incarcare documente ready!")

In [ ]:
# ============================================================
# EXEMPLE DE UTILIZARE
# ============================================================

# --- OPTIUNEA 1: Upload din computer ---
# upload_and_load()

# --- OPTIUNEA 2: Din Google Drive ---
# load_from_drive("Documents/roman.txt")
# load_from_drive("Projects/code.py")

# --- OPTIUNEA 3: Din URL ---
# load_from_url("https://example.com/document.pdf")

# --- OPTIUNEA 4: Incarca toate fisierele dintr-un folder ---
def load_folder(folder_path, extensions=None):
    """Incarca toate fisierele dintr-un folder."""
    folder = Path(folder_path)
    if not folder.exists():
        print(f"Folder not found: {folder_path}")
        return 0
    
    extensions = extensions or ['txt', 'md', 'pdf', 'docx', 'py', 'json', 'csv']
    total = 0
    
    for ext in extensions:
        for filepath in folder.glob(f"**/*.{ext}"):
            try:
                total += load_document_to_memory(filepath)
            except Exception as e:
                print(f"   Eroare la {filepath}: {e}")
    
    print(f"\nTotal incarcat: {total} chunks")
    return total

# Exemplu: incarca toate fisierele .py si .txt din Drive
# load_folder("/content/drive/MyDrive/MyProject", extensions=['py', 'txt'])

print("Exemple disponibile - decommenteaza ce ai nevoie!")

## 5. Demo: Chat cu Memorie Infinita

In [ ]:
# Demo conversatie
print("=" * 60)
print("DEMO: Chat cu Memorie Infinita (LOCAL MODEL)")
print("=" * 60)

messages = [
    "Hello! My name is Alex and I work as a software engineer.",
    "I'm working on a machine learning project for image processing.",
    "What do you know about me and my project?"
]

for msg in messages:
    print(f"\nUSER: {msg}")
    response = chat_with_infinite_context(msg)
    print(f"\nAI: {response[:500]}..." if len(response) > 500 else f"\nAI: {response}")
    print("-" * 40)

In [ ]:
# Verifica memoria stocata
print("\nMEMORY STATS:")
stats = memory.get_stats()
print(f"   Contexts stored: {stats['num_contexts']}")
print(f"   Storage path: {stats['storage_path']}")

# Verifica fisierele in Drive
files = list(Path(STORAGE_PATH).glob("*.pkl"))
print(f"   Files in Drive: {len(files)}")

## 6. Interactive Chat

In [ ]:
# Chat interactiv cu comenzi
print("=" * 60)
print("INTERACTIVE CHAT")
print("=" * 60)
print("""
COMENZI:
  /ls [path]      - listeaza foldere din Drive
  /load <path>    - incarca folder/fisier in memorie
  /search <text>  - cauta in memorie
  /save <text>    - salveaza text in memorie
  /learn <text>   - AI invata/internalizeaza info
  /info           - info sistem
  /memory         - statistici memorie
  /upload         - upload fisier
  /quit           - iesire
""")

import os

while True:
    user_input = input("\nYou: ").strip()
    
    if not user_input:
        continue
    
    # Comenzi speciale
    if user_input.lower() in ['quit', 'exit', '/quit', '/exit', 'q']:
        print("\nGoodbye! Memoria ta e salvata in Drive.")
        break
    
    elif user_input.startswith('/ls'):
        parts = user_input.split(maxsplit=1)
        path = parts[1] if len(parts) > 1 else ""
        full_path = f"/content/drive/MyDrive/{path}".rstrip('/')
        try:
            items = os.listdir(full_path)
            print(f"\n{full_path}/")
            for item in sorted(items)[:30]:
                item_path = os.path.join(full_path, item)
                if os.path.isdir(item_path):
                    print(f"   [DIR] {item}/")
                else:
                    size = os.path.getsize(item_path) / 1024
                    print(f"   {item} ({size:.1f} KB)")
        except Exception as e:
            print(f"Eroare: {e}")
    
    elif user_input.startswith('/load'):
        parts = user_input.split(maxsplit=1)
        if len(parts) < 2:
            print("Utilizare: /load <path>")
        else:
            path = parts[1]
            if not path.startswith('/'):
                path = f"/content/drive/MyDrive/{path}"
            if os.path.isdir(path):
                load_folder(path)
            else:
                load_document_to_memory(path)
    
    elif user_input.startswith('/search'):
        parts = user_input.split(maxsplit=1)
        if len(parts) < 2:
            print("Utilizare: /search <text>")
        else:
            query = parts[1]
            query_emb = embedder.encode(query, convert_to_numpy=True)
            results = memory.retrieve_similar(query_emb, top_k=5)
            print(f"\n=== REZULTATE: '{query}' ===")
            for i, r in enumerate(results):
                meta = r['metadata']
                content = meta.get('content', '')[:100]
                ctx_type = meta.get('type', meta.get('role', '?'))
                print(f"{i+1}. [{r['similarity']:.2f}] ({ctx_type}) {content}...")
    
    elif user_input.startswith('/save '):
        text = user_input[6:].strip()
        if text:
            ctx_id = save_to_memory(text, memory_type="manual")
            print(f"Salvat in memorie (ctx_id={ctx_id}): {text[:50]}...")
        else:
            print("Utilizare: /save <text de salvat>")
    
    elif user_input.startswith('/learn '):
        text = user_input[7:].strip()
        if text:
            # Salveaza ca knowledge
            ctx_id = save_to_memory(text, memory_type="knowledge")
            print(f"Invatat si salvat (ctx_id={ctx_id}): {text[:50]}...")
        else:
            print("Utilizare: /learn <informatie de invatat>")
    
    elif user_input == '/info':
        print(get_system_info())
    
    elif user_input == '/memory':
        stats = memory.get_stats()
        print(f"\n=== MEMORIE ===")
        print(f"Contexte: {stats['num_contexts']}")
        print(f"Storage: {stats['storage_path']}")
        types = {}
        for ctx in memory.contexts.values():
            t = ctx.metadata.get('type', ctx.metadata.get('role', 'unknown'))
            types[t] = types.get(t, 0) + 1
        print("Tipuri:")
        for t, count in sorted(types.items()):
            print(f"   {t}: {count}")
    
    elif user_input == '/upload':
        upload_and_load()
    
    elif user_input.startswith('/'):
        print("Comanda necunoscuta. Comenzi: /ls /load /search /save /learn /info /memory /upload /quit")
    
    else:
        response = chat_with_infinite_context(user_input)
        print(f"\nAI: {response}")

---

## Sumar

**Qwen2.5-32B-Instruct** - Optimal pentru 80GB VRAM

### Specificatii:
- 32 miliarde parametri
- Apache 2.0 (100% gratuit)
- ~64GB VRAM
- Excelent pentru romana si scriere creativa

### FHRSS+FCPE:
- 2M+ tokens context
- 100% recovery la 40% data loss
- Stocare: `/content/drive/MyDrive/FHRSS_FCPE_Colab_OpenSource`

### Functii:
- `upload_and_load()` - upload fisiere
- `load_from_drive("path")` - din Drive
- `load_folder("path")` - tot folderul

---
*Patent: EP25216372.0 (FHRSS - OmniVault)*  
*Author: Vasile Lucian Borbeleac*